In [0]:
import urllib
class HelperClass:
    def __init__(self, uri, filename, location, checkpoint_path, data_catalog, db_name):
        self.base_uri = uri
        self.file_name = filename
        self.location = location
        self.checkpoint = checkpoint_path
        self.catalog_name = data_catalog
        self.db_name = db_name
    
    def download_dataset(self):
        source = self.base_uri + "/" + self.file_name
        target = self.location + "/" + self.file_name
        
        urllib.request.urlretrieve(source, target)
        #response = requests.get(source)
        #if response.status_code != 200:
        #    raise Exception(f"Failed to download dataset from {source}")
        #else:
        #    with open(target, "wb") as file:
        #        file.write(response.content)
    
    
    def create_database(self):
        spark.sql(f"USE CATALOG {self.catalog_name}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {self.db_name}")
        spark.sql(f"USE SCHEMA {self.db_name}")
    
    
    def clean_up(self):
        print("Removing Checkpoints ...")
        dbutils.fs.rm(self.checkpoint, True)
        print("Dropping Database ...")
        spark.sql(f"DROP SCHEMA IF EXISTS {self.db_name} CASCADE")
        print("Removing Dataset ...")
        dbutils.fs.rm(self.location, True)
        print("Done")

    
    def __get_index(self, dir):
        try:
            files = dbutils.fs.ls(dir)
            file = max(f.name for f in files if f.name.endswith('.json'))
            index = int(file.rsplit('.', maxsplit=1)[0])
        except:
            index = 0
        return index+1
  
        
    def process_bronze(self):
        schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

        query = (spark.readStream
                            .format("cloudFiles")
                            .option("cloudFiles.format", "json")
                            .schema(schema)
                            .load(f"{self.location}/kafka-raw")
                            .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))  
                            .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                      .writeStream
                          .option("checkpointLocation", f"{self.checkpoint}/bronze")
                          .option("mergeSchema", True)
                          .partitionBy("topic", "year_month")
                          .trigger(availableNow=True)
                          .table("bronze"))

        query.awaitTermination()
        
        
    def __upsert_data(self, microBatchDF, batch):
        microBatchDF.createOrReplaceTempView("orders_microbatch")
    
        sql_query = """
          MERGE INTO orders_silver a
          USING orders_microbatch b
          ON a.order_id=b.order_id AND a.order_timestamp=b.order_timestamp
          WHEN NOT MATCHED THEN INSERT *
        """

        microBatchDF.sparkSession.sql(sql_query)
        
    def __batch_upsert(self, microBatchDF, batchId):
        window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())
        
        (microBatchDF.filter(F.col("row_status").isin(["insert", "update"]))
                     .withColumn("rank", F.rank().over(window))
                     .filter("rank == 1")
                     .drop("rank")
                     .createOrReplaceTempView("ranked_updates"))

        query = """
            MERGE INTO customers_silver c
            USING ranked_updates r
            ON c.customer_id=r.customer_id
                WHEN MATCHED AND c.row_time < r.row_time
                  THEN UPDATE SET *
                WHEN NOT MATCHED
                  THEN INSERT *
        """

        microBatchDF.sparkSession.sql(query)
        
    
    def __type2_upsert(self, microBatchDF, batch):
        microBatchDF.createOrReplaceTempView("updates")

        sql_query = """
            MERGE INTO books_silver
            USING (
                SELECT updates.book_id as merge_key, updates.*
                FROM updates

                UNION ALL

                SELECT NULL as merge_key, updates.*
                FROM updates
                JOIN books_silver ON updates.book_id = books_silver.book_id
                WHERE books_silver.current = true AND updates.price <> books_silver.price
              ) staged_updates
            ON books_silver.book_id = merge_key 
            WHEN MATCHED AND books_silver.current = true AND books_silver.price <> staged_updates.price THEN
              UPDATE SET current = false, end_date = staged_updates.updated
            WHEN NOT MATCHED THEN
              INSERT (book_id, title, author, price, current, effective_date, end_date)
              VALUES (staged_updates.book_id, staged_updates.title, staged_updates.author, staged_updates.price, true, staged_updates.updated, NULL)
        """

        microBatchDF.sparkSession.sql(sql_query)
    
    def process_orders_silver(self):
        json_schema = "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>"
        
        deduped_df = (spark.readStream
                   .table("bronze")
                   .filter("topic = 'orders'")
                   .select(F.from_json(F.col("value").cast("string"), json_schema).alias("v"))
                   .select("v.*")
                   .withWatermark("order_timestamp", "30 seconds")
                   .dropDuplicates(["order_id", "order_timestamp"]))
        
        query = (deduped_df.writeStream
                   .foreachBatch(self.__upsert_data)
                   .outputMode("update")
                   .option("checkpointLocation", f"{self.checkpoint}/orders_silver")
                   .trigger(availableNow=True)
                   .start())

        query.awaitTermination()

        
    def process_customers_silver(self):
        
        schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time timestamp"
        
        df_country_lookup = spark.read.json(f"{dataset_bookstore}/country_lookup")

        query = (spark.readStream
                          .table("bronze")
                          .filter("topic = 'customers'")
                          .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
                          .select("v.*")
                          .join(F.broadcast(df_country_lookup), F.col("country_code") == F.col("code") , "inner")
                       .writeStream
                          .foreachBatch(self.__batch_upsert)
                          .outputMode("update")
                          .option("checkpointLocation", f"{self.checkpoint}/customers_silver")
                          .trigger(availableNow=True)
                          .start()
                )

        query.awaitTermination()
    
    def process_books_silver(self):
        schema = "book_id STRING, title STRING, author STRING, price DOUBLE, updated TIMESTAMP"

        query = (spark.readStream
                        .table("bronze")
                        .filter("topic = 'books'")
                        .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
                        .select("v.*")
                     .writeStream
                        .foreachBatch(self.__type2_upsert)
                        .option("checkpointLocation", f"{self.checkpoint}/books_silver")
                        .trigger(availableNow=True)
                        .start()
                )

        query.awaitTermination()
        
    def process_current_books(self):
        spark.sql("""
            CREATE OR REPLACE TABLE current_books
            AS SELECT book_id, title, author, price
               FROM books_silver
               WHERE current IS TRUE
        """)

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/dataset/mfi"))

In [0]:
%sh
rm /Volumes/workspace/default/dataset/mfi/mfi_csv_250411.csv.gz

In [0]:
df = spark.read.format("csv").option("header", True).option("inferSchema", "true").option("encoding", "UTF-16").option("sep", "\t").load("/Volumes/workspace/default/dataset/mfi/mfi_csv_250411.csv.gz")
display(df)

In [0]:
from datetime import datetime, timedelta
# Erstelle ein Text-Widget, falls noch nicht vorhanden.
dbutils.widgets.text("execution_date", "", "Ausführungstag (Format: YYYY-MM-DD)")

# Lese den Parameter aus:
execution_date = dbutils.widgets.get("execution_date")
execution_date = "2025-04-12"
if execution_date == "":
    reference_date = datetime.today()
else:
    reference_date = datetime.strptime(execution_date, "%Y-%m-%d")

print("Das abzufragende Datum:", reference_date.strftime("%Y-%m-%d"))

#berechnun vortag
vortag = reference_date - timedelta(days=1)
filename= f"mfi_csv_{vortag.strftime('%y%m%d')}.csv.gz"
print("Das abzufragende Datum (Vortag):", vortag.strftime("%Y-%m-%d"))
data_source_baseuri=f"https://www.ecb.europa.eu/stats/money/mfi/general/html/dla/mfi_MID"
dataset_target = '/Volumes/workspace/default/dataset/mfi'
checkpoint_path = "dbfs:/Volumes/workspace/default/dataset/checkpoints"
data_catalog = 'datashop_catalog'
db_name = "datashop_mfi"

mfidatastore = HelperClass(data_source_baseuri,filename, dataset_target, checkpoint_path, data_catalog, db_name)
mfidatastore.download_dataset()
mfidatastore.create_database()